In [2]:
import librosa
import numpy as np
import scipy.signal
from python_speech_features import mfcc, delta
import pywt

In [3]:
audio_path = 'crowded.wav'
y, sr = librosa.load(audio_path, sr=None)

Fitur Basis waktu (Time Domain Features)

In [ ]:
# 1 . Zero Crossing Rate (ZCR)
zero_crossing_rate = librosa.feature.zero_crossing_rate(y)
print("zero-crossing rate :", np.mean(zero_crossing_rate))

# 2. energy
energy = np.sum(y**2)
print("energy :", energy)

# 3. root mean square (RMS)
rms = np.sqrt(np.mean(y**2))
print("RMS :", rms)

# 4. Autocorrelation
# autocorr = np.correlate(y, y, mode='full')[len(y)-1:]
# print("Autocorrelation :", autocorr[:10])

zero-crossing rate : 0.027044054513518715
energy : 818.36145
RMS : 0.009729765


Fitur Basis Frekuensi (Frequency-Domain Features)

In [ ]:
# 1. Spectral Centroid
spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
print("Spectral Centroid:", np.mean(spectral_centroid))

# 2. Spectral Bandwidth
spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
print("Spectral Bandwidth:", np.mean(spectral_bandwidth))

# 3. Spectral Contrast
spectral_contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
print("Spectral Contrast:", np.mean(spectral_contrast, axis=1))  # Per band

# 4. Spectral Roll-off
spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.85)
print("Spectral Roll-off:", np.mean(spectral_rolloff))

Spectral Centroid: 1819.2728726566636
Spectral Bandwidth: 2904.567104519785
Spectral Contrast: [15.62649313  9.94956738 13.68359303 14.70670567 16.40526023 15.81389109
 29.69028657]
Spectral Roll-off: 3731.7028140547263


Fitur Basis Representasi Frekuensi (Cepstral Features)

In [ ]:
# 1. MFCCs
mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)  # 13 koefisien MFCC umum digunakan
print("MFCCs:", np.mean(mfccs, axis=1))  # Per koefisien

# 2. LPC
def lpc_coefficients(signal, order):
    """
    Menghitung koefisien LPC dengan pendekatan polinomial.
    """
    return np.polyfit(np.arange(len(signal)), signal, order)

lpc_order = 12 # Order LPC biasanya antara 10-16 untuk suara
lpc_features = lpc_coefficients(y, lpc_order)
print("LPC Coefficients:", lpc_features)

# 3. PLP
plp_features = delta(mfccs, 2)  # Perbedaan tingkat kedua dari MFCC
print("PLP Features:", np.mean(plp_features, axis=1))

MFCCs: [-459.58884    155.96422     18.133879    30.913448     4.901187
    6.4847217    3.5400789   -2.0726168    1.8635439   -5.095873
   -4.0759206   -1.4279852   -2.1847205]
LPC Coefficients: [ 8.91072330e-83 -4.69163427e-75  1.08394757e-67 -1.44459022e-60
  1.22802241e-53 -6.95264751e-47  2.65756147e-40 -6.80381153e-34
  1.13206054e-27 -1.15096423e-21  6.33687159e-16 -1.44158160e-10
  6.18393617e-06]
PLP Features: [157.09987    145.87274     80.39293    -31.219168    -5.361633
  -6.7333236   -1.4632624   -2.4837723   -1.8255254   -0.46502012
  -0.44286403   0.77135056   0.3025665 ]


Fitur Basis Waktu Frekuensi (Time-Frequency Features)

In [ ]:
# 1. STFT
stft = np.abs(librosa.stft(y, n_fft=1024, hop_length=512))
print("STFT Shape:", stft.shape)  # (frekuensi, frame)

stft_mean = np.mean(stft, axis=1) # Ekstraksi nilai rata-rata STFT sebagai fitur
print("STFT Mean:", stft_mean)

# 2. Wavelet Transform
scales = np.arange(1, 128)
coefficients, _ = pywt.cwt(y, scales, 'morl')

wavelet_mean = np.mean(coefficients, axis=1) # Ambil rata-rata koefisien sebagai fitur
print("Wavelet Transform Mean:", wavelet_mean)

# 3. Chrome Features
chroma = librosa.feature.chroma_stft(y=y, sr=sr)
print("Chroma Shape:", chroma.shape)

chroma_mean = np.mean(chroma, axis=1) # Ambil nilai rata-rata sebagai fitur
print("Chroma Mean:", chroma_mean)

STFT Shape: (513, 16884)
STFT Mean: [1.81969017e-01 4.99266773e-01 9.09386754e-01 1.00603044e+00
 1.06685436e+00 9.77582455e-01 8.22353363e-01 7.43677557e-01
 7.55597293e-01 7.52331436e-01 6.92171097e-01 6.20337665e-01
 5.55278301e-01 4.83502001e-01 3.95260811e-01 3.10579032e-01
 2.58748174e-01 2.23605767e-01 2.03985780e-01 1.94574550e-01
 1.71157703e-01 1.52366817e-01 1.38793796e-01 1.33816957e-01
 1.35128930e-01 1.19017035e-01 1.08471788e-01 1.06587537e-01
 1.00001059e-01 9.64193121e-02 9.91354808e-02 9.61317942e-02
 9.19753090e-02 8.56701657e-02 7.95183033e-02 7.21152648e-02
 6.54470548e-02 5.99775799e-02 5.56509607e-02 5.30332737e-02
 4.98616211e-02 4.52327318e-02 4.13195193e-02 3.93682607e-02
 3.96080762e-02 4.01553921e-02 4.13933247e-02 4.16339561e-02
 4.22791541e-02 4.46720198e-02 4.51572873e-02 4.50092182e-02
 4.63766940e-02 4.40492518e-02 4.12673317e-02 4.03172076e-02
 3.92147787e-02 3.85348387e-02 3.63852009e-02 3.32534760e-02
 3.20321210e-02 3.12717929e-02 3.10387202e-02 3.1